# Chapitre 4 - Exercices 4.1 à 4.6

Ce notebook traite les six premiers exercices du chapitre 4, centrés sur les formats
d'encodage et l'évolution des schémas :

- **4.1** : compter les octets à la main dans chaque format (calcul) ;
- **4.2** : varints, zigzag et encodage différentiel d'horodatages (calcul et
  mini-lab) ;
- **4.3** : grands identifiants et Base64, les pièges des formats textuels (calcul) ;
- **4.4** : écrire un vérificateur d'évolution de schémas Protocol Buffers (mini-lab) ;
- **4.5** : décoder un enregistrement Avro à la main et prédire la résolution
  (raisonnement) ;
- **4.6** : registre de schémas, compatibilité transitive et rétention (raisonnement
  et code).

In [1]:
import json
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ddia import encoding as E
from ddia.encoding import PbField, PbMessage
from ddia.plotting import (AMBER, GRAY, GREEN, INDIGO, LIGHT, ROSE, SKY, SLATE, TEAL, VIOLET, COLORS,
                           annotate_bars, finish, set_style, subtitle)

set_style()
plt.rcParams["axes.titlepad"] = 16
pd.set_option("display.precision", 3)
pd.set_option("display.max_colwidth", 120)
rng = np.random.default_rng(0)

## Exercice 4.1 : compter les octets à la main

**Énoncé.** On encode l'enregistrement
`{"userName": "Zoé", "favoriteNumber": -200, "interests": ["vélo"]}` (attention : `é` est
un caractère non ASCII).

1. Combien d'octets en JSON compact ? en MessagePack ?
2. Combien d'octets en Protocol Buffers avec le schéma du cours (`int64
   favorite_number = 2`) ? Et si le champ est déclaré `sint64` ?
3. Combien d'octets en Avro avec le schéma `Person` du cours ?
4. Classer les formats et expliquer l'écart entre `int64` et `sint64`.

5. Combien d'octets en Thrift BinaryProtocol et CompactProtocol ?

**Rappels.** UTF-8 code `é` sur 2 octets. MessagePack : chaîne courte = 1 octet
(`0xa0 | longueur`) + contenu ; entier de -32 768 à 32 767 hors des plages courtes : 1
octet de type + 2. Protobuf : clé d'un octet (numéro < 16), longueurs et entiers en
varint ; un `int64` négatif est écrit en complément à deux sur 64 bits, donc en 10 octets
de varint ; `sint64` utilise le zigzag. Avro : longueurs et entiers en zigzag, indice de
branche d'union, tableau = compte + éléments + 0 (cours 1 à 3, sections *Binary encoding*,
*Thrift and Protocol Buffers*, *Avro*).

**Solution.**

*1. JSON.* `{"userName":"Zoé","favoriteNumber":-200,"interests":["vélo"]}` compte 61
caractères, dont 2 `é` de 2 octets : **63 octets**. *MessagePack* : en-tête de map (1) ;
`userName` (1 + 8) ; `Zoé` (1 + 4) ; `favoriteNumber` (1 + 14) ; -200 ne tient pas dans
l'entier négatif court (-32 à -1) : `0xd1` + 2 octets (3) ; `interests` (1 + 9) ; tableau
d'un élément (1) ; `vélo` (1 + 5). Total : 1 + 9 + 5 + 15 + 3 + 10 + 1 + 6 = **50 octets**.

*2. Protobuf.* Champ 1 : clé + longueur + 4 octets = 6. Champ 2 en `int64` : -200 en
complément à deux a ses bits de poids fort à 1, le varint fait **10 octets**, plus la
clé : 11. Champ 3 : clé + longueur + 5 = 7. Total **24 octets**. En `sint64` : zigzag(-200)
= 399, qui tient sur 2 octets de varint (399 < 16 384) : 1 + 2 = 3, total **16 octets**.

*3. Avro.* Longueur 4 en zigzag (1 octet `08`) + 4 octets ; indice de branche 1 (1 octet) +
zigzag(-200) = 399 sur 2 octets ; tableau : compte (1) + longueur (1) + 5 octets + fin
(1). Total : 5 + 3 + 8 = **16 octets**.

*4.* Avro (16) = Protobuf `sint64` (16) < Protobuf `int64` (24) < MessagePack (50) < JSON
(63). Un `int64` négatif coûte toujours 10 octets en protobuf : pour un champ qui peut être
négatif, `sint64` est le bon choix.

*5. Thrift.* BinaryProtocol : chaque en-tête de champ fait 3 octets, les longueurs 4, un
`i64` 8 : (3 + 4 + 4) + (3 + 8) + (3 + 1 + 4 + 4 + 5) + 1 (fin) = **40 octets**.
CompactProtocol : (1 + 1 + 4) + (1 + 2) + (1 + 1 + 1 + 5) + 1 = **18 octets** (les entiers
sont toujours en zigzag).